# Unified Model Embeddings

## Building Environment

In [1]:
!pip install -q nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama gradio tabulate soundfile librosa seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 14.9 MB/s eta 0:00:0000:01


In [2]:
!mkdir -p ARL
base="https://github.com/James-Tiny-Tjib/ARL/raw/main/ARL"
!curl -L -o "ARL/Airport_Noise_Dataset.zip" "$base/Airport%20Noise%20Dataset-20260603T141901Z-3-001.zip"
!curl -L -o "ARL/DroneAudioDataset.zip" "$base/DroneAudioDataset.zip"
!curl -L -o "ARL/best_cnn.pt" "$base/best_cnn.pt"
!curl -L -o "ARL/drone_demo_dataset.zip" "$base/drone_demo_dataset.zip"
!curl -L -o "ARL/drone_multi_classifier.pt" "$base/drone_multi_classifier.pt"
!curl -L -o "ARL/resnet50_drone_weights.pth" "$base/resnet50_drone_weights.pth"
!curl -L -o "ARL/sensor_client.pt" "$base/sensor_client.pt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  293k    0  293k    0     0   797k      0 --:--:-- --:--:-- --:--:--  798k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  286M  100  286M    0     0  34.2M      0  0:00:08  0:00:08 --:--:-- 39.4M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  394k  100  394k    0     0   654k      0 --:--:-- --:--:-- --:--:--  654k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   To

In [3]:
import os, io, csv, time, random, threading, subprocess, asyncio
import base64, pickle, tempfile, shutil
from collections import Counter, defaultdict
from pathlib import Path
from itertools import cycle
import copy

import numpy as np
import torch
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import r2_score, accuracy_score, classification_report
import librosa
import soundfile as sf
import pandas as pd

import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from io import BytesIO

import nest_asyncio
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
BASE_DIR          = "./ARL"
AUDIO_MODEL_PATH  = f"{BASE_DIR}/drone_multi_classifier.pt"
RF_MODEL_PATH     = f"{BASE_DIR}/sensor_client.pkl"
VISUAL_MODEL_PATH = f"{BASE_DIR}/resnet50_drone_weights.pth"
MULTI_MODAL_MODEL_PATH = f"{BASE_DIR}/xgboostmodel.pkl"
SNAPSHOT_CSV_PATH = "./ARL/snapshot_log.csv"
DEMO_IMAGES_DIR   = "./drone_demo_dataset"
AUDIO_DATASET_DIR = "./DroneAudioDataset/Multiclass_Drone_Audio"
VISUAL_DATASET_DIR = "./drone_demo_dataset"
RF_MODEL_PATH_PT = f"{BASE_DIR}/sensor_client.pt"

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 4000
NUM_SENSORS = 23
MCP_PORT    = 8001

print(f"Device: {DEVICE}")

Device: cuda


## Data Preparation

In [ ]:
# Drone audio dataset
ZIP_PATH = f"{BASE_DIR}/DroneAudioDataset.zip"
if os.path.exists(ZIP_PATH):
    print(f"Unzipping {ZIP_PATH}...")
    # FIX: Extract into "./DroneAudioDataset" instead of "."
    os.system(f'unzip -o "{ZIP_PATH}" -d "./DroneAudioDataset"')
elif not os.path.exists(AUDIO_DATASET_DIR):
    print("Zip not found, cloning from GitHub...")
    os.system("git clone https://github.com/saraalemadi/DroneAudioDataset.git")
else:
    print("DroneAudioDataset already exists, skipping.")

# Drone Demo Dataset
DEMO_PATH = f"{BASE_DIR}/drone_demo_dataset.zip"
if os.path.exists(DEMO_PATH):
    print(f"Unzipping {DEMO_PATH}...")
    os.system(f'unzip -o "{DEMO_PATH}" -d "./drone_demo_dataset"')
else:
    print(" already exists, skipping.")


# Background (airport) noise
BG_DIR      = f"{AUDIO_DATASET_DIR}/bg noise"
AIRPORT_ZIP = f"{BASE_DIR}/Airport_Noise_Dataset.zip"
os.makedirs(BG_DIR, exist_ok=True)
existing = len([f for f in os.listdir(BG_DIR) if f.endswith('.wav')]) if os.path.exists(BG_DIR) else 0
if existing > 0:
    print(f"bg noise already has {existing} .wav files, skipping.")
elif os.path.exists(AIRPORT_ZIP):
    tmp = tempfile.mkdtemp()
    os.system(f'unzip -o "{AIRPORT_ZIP}" -d "{tmp}"')
    count = 0
    for root, _, files in os.walk(tmp):
        for fn in files:
            if fn.endswith('.wav'):
                shutil.copy(os.path.join(root, fn), os.path.join(BG_DIR, fn))
                count += 1
    shutil.rmtree(tmp)
    print(f"Copied {count} .wav files into bg noise.")
else:
    print("No airport noise zip found.")

In [12]:
# Dataset summary
if os.path.exists(AUDIO_DATASET_DIR):
    for item in sorted(os.listdir(AUDIO_DATASET_DIR)):
        full = os.path.join(AUDIO_DATASET_DIR, item)
        n = len([f for f in os.listdir(full) if f.endswith('.wav')]) if os.path.isdir(full) else 0
        print(f"  {item}/  ({n} .wav files)")

  bebop_1/  (666 .wav files)
  bg noise/  (118 .wav files)
  membo_1/  (666 .wav files)
  unknown/  (10372 .wav files)


## Model Architectures

In [13]:
# ── Audio: DroneCNN ──────────────────────────────────────────────────────────
class DroneCNN(nn.Module):
    """3-class mel-spectrogram classifier: Mambo / Bebop / Background."""
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 3),
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))


# ── RF: IQCNN + Client ───────────────────────────────────────────────────────

class IQCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(IQCNN, self).__init__()

        self.layer_dims=[]
        self.layers=nn.ModuleList()

        # Define convolutional layers here .......................................
        self.layers.append(nn.Conv1d(in_channels=2, out_channels=8, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=8, out_channels=16, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3,bias=False))

        self.conv_num=len(self.layers)
        for i in range(self.conv_num):
          in_ch=self.layers[i].in_channels
          ker_sz=self.layers[i].kernel_size[0]
          self.layer_dims.append(in_ch*ker_sz)

        #Define linear layers here .....................................................
        self.layers.append(nn.Linear(64, 256,bias=False))
        self.layers.append(nn.Linear(256, num_classes,bias=False))

        for i in range(self.conv_num,len(self.layers)):
          self.layer_dims.append(self.layers[i].in_features)
        self.layer_dims.append(num_classes)
        # print(self.layer_dims)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        for i in range(self.conv_num):
          nn.init.xavier_uniform_(self.layers[i].weight)
        for i in range(self.conv_num,len(self.layers)-1):
            nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='relu')

        nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='linear')

    def forward(self, x):

        for i in range(self.conv_num):
          x=F.relu(self.layers[i](x))


        x = self.global_avg_pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)

        for i in range(self.conv_num,len(self.layers)-1):
          x=F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (batch_size, output_shape)

        return x#F.log_softmax(x, dim=1)  # Use log_softmax for classification

def build_model(num_classes=2,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)

class Client:
  def __init__(self,dataset,num_classes=2,device='cpu'):
      self.device=device
      self.model = build_model(num_classes=num_classes).to(self.device)
      self.dataset =dataset
      self.dataloader = cycle(t.utils.data.DataLoader(self.dataset, batch_size=512, shuffle=True))
      self.cross_loss = nn.CrossEntropyLoss()
      self.optimizer = t.optim.Adam(self.model.parameters(), lr=0.001)

  def load_model(self,state_dict):
    self.model.load_state_dict(state_dict)

  def load_model_from_file(self,filename):
    state_dict=read_data(filename)
    self.load_model(state_dict)

  def save_model(self,filename):
    save_data(self.model.state_dict(),filename)

  def client_loss(self,pred,y):
    return self.cross_loss(pred,y)

  def evaluate(self,test_loader):
    correct,total=0,0
    with t.no_grad():
        for data in test_loader:
            x, y = data
            x=x.to(self.device)
            y=y.to(self.device)
            output = self.model(x)
            for idx, i in enumerate(output):
                if t.argmax(i) == y[idx]:
                    correct +=1
                total +=1
    print(f'accuracy: {round(correct/total, 3)}')
    return round(correct/total, 3)


  def train_batch(self):
    x,y = next(self.dataloader)
    x=x.to(self.device)
    y=y.to(self.device)
    self.optimizer.zero_grad()
    pred = self.model(x)
    loss=self.client_loss(pred,y)
    loss.backward()
    self.optimizer.step()
    return loss.item()

  def train(self,test_loader,epochs=int(1e3),echo=True,eval_acc=False):
    running_loss=0
    for epoch in range(epochs):
      epoch_loss = self.train_batch()
      running_loss+=epoch_loss

      if echo:
        if epoch%int(epochs/10) ==0 and epoch!=0:
          print("epoch:"+str(epoch)+" loss:"+ str(running_loss/int(epochs/10)))
          running_loss=0
          if eval_acc:
            self.evaluate(test_loader)
        # if epoch%(echo*10)==0 and epoch!=0:
        #   self.evaluate()

    return running_loss/int(epochs/10)


# ── Visual: ResNet50 ─────────────────────────────────────────────────────────
def build_visual_model(num_classes=1):
    """ResNet50 binary drone detector."""
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

## Data Loaders

In [14]:
AUDIO_DATA_PATHS = {
    "mambo":      f"{AUDIO_DATASET_DIR}/membo_1",
    "bebop":      f"{AUDIO_DATASET_DIR}/bebop_1",
    "background": f"{AUDIO_DATASET_DIR}/bg noise",
}


class DroneAudioDataset(Dataset):
    """Loads .wav files and returns (mel-spectrogram tensor, label) pairs."""
    def __init__(self, files, labels, sr=22050, duration=1.0):
        self.files, self.labels, self.sr, self.duration = files, labels, sr, duration

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        y, sr = librosa.load(self.files[idx], duration=self.duration, sr=self.sr)
        n = int(self.sr * self.duration)
        if len(y) < n:
            y = np.pad(y, (0, n - len(y)))
        spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        spec_db = (spec_db - spec_db.mean()) / (spec_db.std() + 1e-8)
        return torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0), torch.tensor(self.labels[idx], dtype=torch.long)


def get_audio_loaders(data_paths=AUDIO_DATA_PATHS, max_per_class=800, batch_size=32, test_split=0.2):
    """Returns (train_loader, test_loader) for the drone audio dataset."""
    all_files, all_labels = [], []
    for label, (name, path) in enumerate(data_paths.items()):
        if not os.path.exists(path):
            print(f"Warning: {path} not found, skipping {name}")
            continue
        files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.wav')][:max_per_class]
        all_files.extend(files)
        all_labels.extend([label] * len(files))
    X_tr, X_te, y_tr, y_te = train_test_split(all_files, all_labels, test_size=test_split)
    return (
        DataLoader(DroneAudioDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),
        DataLoader(DroneAudioDataset(X_te, y_te), batch_size=batch_size),
    )


def create_synthetic_data(noise_std,grid_sz,grid_step,num_datapoints=int(1e3)):
  seq_sz=128
  adversary_pwr=1
  num_classes=2 # 0: good 1: adversary

  grid_dist=[i for i in range(int(grid_sz/grid_step))]

  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  data=[]
  labels=[]

  for c in range(num_classes):
    for d in grid_dist:
      for _ in range(num_datapoints):
        p=np.random.uniform(0, 1)
        rx_i, rx_q = np.array([]), np.array([])
        bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
        qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
        qpsk_sig=scale_qpsk*qpsk_sig

        merge_vec=np.random.uniform(0,1,size=seq_sz)
        merge_vec=merge_vec<=p

        sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

        if c==1:
          mp = np.array([-3, -1, 1, 3])
          adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
          adv_sig=adv_sig*scale_16qam
          if d!=0:
            sig=(adversary_pwr/d)*adv_sig+sig
          else:
            sig=adv_sig+sig

        noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
        sig=sig+noise

        data.append(np.array([np.real(sig),np.imag(sig)]))
        labels.append(c)

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset


def generate_sensor_rf_sample(noise_std,p,c,d):
  """
  p: bpsk qpsk prob
  c: 1 adversary
  d: distance
  """
  seq_sz=128
  adversary_pwr=1
  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
  qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
  qpsk_sig=scale_qpsk*qpsk_sig

  merge_vec=np.random.uniform(0,1,size=seq_sz)
  merge_vec=merge_vec<=p

  sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

  if c==1:
    mp = np.array([-3, -1, 1, 3])
    adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
    adv_sig=adv_sig*scale_16qam
    if d!=0:
      sig=(adversary_pwr/d)*adv_sig+sig
    else:
      sig=adv_sig+sig

  noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
  sig=sig+noise

  sample=t.from_numpy(np.array([np.real(sig),np.imag(sig)])).to(device)
  sample=sample.to(t.float32)
  return sample.unsqueeze(0)

visual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def get_visual_loaders(data_dir=VISUAL_DATASET_DIR, batch_size=64, test_split=0.2, val_split=0.1):
    """
    Returns (train_loader, val_loader, test_loader) for the drone visual dataset.
    Expects an ImageFolder layout:  data_dir/birds/  data_dir/drones/  data_dir/planes/
    Labels are remapped to binary:  drone=1, everything else=0.
    """
    from torchvision import datasets
    from torch.utils.data import random_split

    dataset = datasets.ImageFolder(root=data_dir, transform=visual_transforms)
    print(f"Visual dataset classes: {dataset.class_to_idx}")

    # Remap to binary:  drone → 1, bird/plane → 0
    new_samples, new_targets = [], []
    drone_idx = dataset.class_to_idx.get("drones", dataset.class_to_idx.get("drone", -1))
    for path, old_label in dataset.samples:
        binary = 1 if old_label == drone_idx else 0
        new_samples.append((path, binary))
        new_targets.append(binary)
    dataset.samples = new_samples
    dataset.targets = new_targets

    # Split
    total   = len(dataset)
    val_sz  = int(val_split * total)
    test_sz = int(test_split * total)
    train_sz = total - val_sz - test_sz
    print(f"Visual split -> Train: {train_sz} | Val: {val_sz} | Test: {test_sz}")

    train_ds, val_ds, test_ds = random_split(
        dataset, [train_sz, val_sz, test_sz],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

## Load Pretrained Models

In [21]:
def save_data(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def read_data(path):
    class _CPU(pickle.Unpickler):
        def find_class(self, module, name):
            if module == 'torch.storage' and name == '_load_from_bytes':
                return lambda b: torch.load(io.BytesIO(b), map_location='cpu', weights_only=False)
            return super().find_class(module, name)
    with open(path, 'rb') as f:
        return _CPU(f).load()


def load_audio_model(path=AUDIO_MODEL_PATH):
    model = DroneCNN().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    print(f"Loaded audio model from {path}")
    return model


def load_rf_model(path=RF_MODEL_PATH):
    rf_weights = read_data(path)
    model = build_model(num_classes=2)
    model.load_state_dict(rf_weights)
    print(f"Loaded RF model from {path}")
    return model

def load_rf_model_pt(path=RF_MODEL_PATH_PT):
    rf_weights = torch.load(path, map_location=DEVICE)
    model = build_model(num_classes=2)
    model.load_state_dict(rf_weights)
    print(f"Loaded RF model from {path}")
    return model


def load_visual_model(path=VISUAL_MODEL_PATH):
    model = build_visual_model().to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
    model.eval()
    print(f"Loaded visual model from {path}")
    return model

def load_multi_modal_model(path = MULTI_MODAL_MODEL_PATH):
    model = read_data(path)
    print(f"Loaded Multi_Modal from {path}")
    return model


# AU_MODEL  = load_audio_model()
# RF_MODEL    = load_rf_model()
# VS_MODEL = load_visual_model()
# FS_MODEL = load_multi_modal_model()

cuda


## Training

In [17]:
def train_audio_model(save_path=AUDIO_MODEL_PATH, epochs=10, batch_size=32):
    """Train DroneCNN on drone audio dataset and save weights."""
    train_loader, test_loader = get_audio_loaders(batch_size=batch_size)
    model = DroneCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}")
    # Evaluate
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    print(f"Test accuracy: {correct/total:.3f}")
    torch.save(model.state_dict(), save_path)
    print(f"Saved -> {save_path}")
    return model


def train_rf_client(save_path=RF_MODEL_PATH, epochs=2000):
    model = None

    # Free GPU memory from other models before training
    train_loader,test_loader,trainset,testset=create_synthetic_data(noise_std=0.1,grid_sz=80,grid_step=4,num_datapoints=int(1e4))
    device= 'cuda' if t.cuda.is_available() else 'cpu'
    print(device)

    rf_client= Client(trainset,device=device)
    rf_client.train(test_loader,epochs=epochs,echo=True,eval_acc=True)
    rf_client.evaluate(test_loader)
    rf_client.save_model(RF_MODEL_PATH)
    return rf_client

def train_visual_model(save_path=VISUAL_MODEL_PATH, epochs=5, batch_size=64, lr=0.001):
    """
    Train ResNet50 (transfer-learned) binary drone detector and save weights as .pth.
    Freezes the backbone and only trains the final FC layer, matching the
    original Colab training procedure.
    """
    train_loader, val_loader, test_loader = get_visual_loaders(batch_size=batch_size)

    # Build model with pretrained ImageNet backbone
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 1)
    model = model.to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr)

    best_val_acc = 0.0

    for epoch in range(epochs):
        # ── Training ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE).float().view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validation ──
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE).float().view(-1, 1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/val_total:.4f}  Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    # ── Test ──
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE).float().view(-1, 1)
            preds = (torch.sigmoid(model(inputs)) >= 0.5).float()
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
    print(f"Test Accuracy: {test_correct/test_total:.4f}")

    # Save as .pth state_dict (compatible with load_visual_model)
    torch.save(model.state_dict(), save_path)
    print(f"Saved visual model -> {save_path}")
    return model

# Uncomment to retrain from scratch:
# train_audio_model()
# train_rf_client()
# train_visual_model()

In [24]:
from __future__ import annotations
from typing import Dict, List
import torch
import torch.nn as nn
import torch.nn.functional as F

MODALITIES: List[str] = ["rf", "audio", "visual"]
IN_DIMS: Dict[str, int] = {"rf": 256, "audio": 128, "visual": 2048}
D_MODEL = 256


class ModalityEncoder(nn.Module):
    def __init__(self, base, head, in_dim, d=D_MODEL, freeze=True):
        super().__init__()
        self.base, self.freeze = base, freeze
        self.proj = nn.Linear(in_dim, d)
        self._feat = None
        head.register_forward_pre_hook(lambda m, a: setattr(self, "_feat", a[0]))
        if freeze:
            for p in self.base.parameters():
                p.requires_grad = False
            self.base.eval()
 
    def train(self, mode=True):
        super().train(mode)
        if self.freeze:
            self.base.eval()
        return self
 
    def forward(self, x):
        if self.freeze:
            with torch.no_grad():
                _ = self.base(x)
        else:
            _ = self.base(x)
        return self.proj(self._feat)


class FixedThreatModel(nn.Module):
    def __init__(self, rf_model, audio_model, visual_model,
                 d=D_MODEL, modalities=MODALITIES, freeze=True):
        super().__init__()
        self.modalities = modalities
        self.encoders = nn.ModuleDict({
            "rf":     ModalityEncoder(rf_model,     rf_model.layers[-1],       IN_DIMS["rf"],     d, freeze),
            "audio":  ModalityEncoder(audio_model,  audio_model.fc_layers[-1], IN_DIMS["audio"],  d, freeze),
            "visual": ModalityEncoder(visual_model, visual_model.fc,           IN_DIMS["visual"], d, freeze),
        })
    def forward(self, inputs: Dict[str, torch.Tensor]):
        tokens = torch.stack([self.encoders[m](inputs[m]) for m in self.modalities], dim=1)  # (B,M,d)
        return tokens


def test_model_with_data(mm_model, batch_size):
    mm_model.eval()
    _, au_test_loader = get_audio_loaders(batch_size=batch_size)
    _, _, vs_test_loader = get_visual_loaders(batch_size=batch_size)
    _,rf_test_loader,_, _ =create_synthetic_data(noise_std=0.1,grid_sz=80,grid_step=4,num_datapoints=int(1e4))

    vs_inputs, vs_labels = next(iter(vs_test_loader))
    au_inputs, au_labels = next(iter(au_test_loader))
    rf_inputs, rf_labels = next(iter(rf_test_loader))

    inputs_dict = {
        "visual": vs_inputs[:batch_size].to(DEVICE),
        "audio": au_inputs[:batch_size].to(DEVICE),
        "rf": rf_inputs[:batch_size].to(DEVICE)
    }

    with torch.no_grad():
        output_embeddings = mm_model(inputs_dict)

    print(f"Visual Inputs: {vs_inputs.shape}")
    print(f"Audio Inputs:  {au_inputs.shape}")
    print(f"RF Inputs:     {rf_inputs.shape}")
    print(f"Output Tensor: {output_embeddings.shape}")

    
    


if __name__ == "__main__":
    mm_model = FixedThreatModel(
        load_rf_model_pt(), 
        load_audio_model(), 
        load_visual_model(), 
        d = D_MODEL,
        modalities = MODALITIES,
        freeze = True
    ).to(DEVICE)

    mm_model = torch.compile(mm_model)

    test_model_with_data(mm_model, 2)

    

    
    
    



    
    

    
    

Loaded RF model from ./ARL/sensor_client.pt
Loaded audio model from ./ARL/drone_multi_classifier.pt
Loaded visual model from ./ARL/resnet50_drone_weights.pth
Visual dataset classes: {'birds': 0, 'drones': 1, 'planes': 2}
Visual split -> Train: 210 | Val: 30 | Test: 60
(400000, 2, 128) (400000,)
(280000, 2, 128) (120000, 2, 128) (280000,) (120000,)


W0706 16:54:51.551000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Visual Inputs: torch.Size([2, 3, 224, 224])
Audio Inputs:  torch.Size([2, 1, 128, 44])
RF Inputs:     torch.Size([10, 2, 128])
Output Tensor: torch.Size([2, 3, 256])
